# 📓 Multimodal AI Module 3: Visual Instruction Tuning with Frozen LLMs on MS-COCO
Welcome to Module 3 of Multimodal AI! In this tutorial, we will implement and train a **LLaVA-style Vision-Language Model (VLM)** using a real subset of the **MS-COCO Captions** dataset [cite: 16].

---

## 💡 1. The LLaVA Architecture: Frozen Foundation Backbones
Training end-to-end multimodal models from scratch is computationally expensive [cite: 16]. Modern Large Multimodal Models (LMMs) like **LLaVA**, **Flamingo**, **Frozen**, and **BLIP-2/3** solve this by connecting pre-trained, frozen unimodal backbones [cite: 16]:

1. **Frozen Vision Encoder:** A Vision Transformer (e.g., **CLIP-ViT**) extracts patch-level feature vectors $Z_v \in \mathbb{R}^{P \times d_v}$ [cite: 16].
2. **Trainable Multimodal Projector:** A lightweight 2-layer MLP projection network maps visual patch embeddings into the language model's word embedding dimension: $H_v = \text{MLP}(Z_v) \in \mathbb{R}^{P \times d_{\text{LLM}}}$ [cite: 16].
3. **Frozen Language Model (LLM):** A decoder-only language model (e.g., GPT-2 or LLaMA) treats the projected visual tokens as a soft prefix prepended to text token embeddings: $H = [H_v; H_t]$ [cite: 16].

```
Input Image ──► [Frozen CLIP-ViT] ──► Features Z_v ──► [Trainable Projector W] ──► Visual Tokens H_v ┐
                                                                                                      ├──► [Frozen LLM] ──► Text Output
Prompt Text ────────────────────────► Tokenizer  ──► Word Embeddings H_t ────────────────────────────┘
```

### Stage 1: Feature Alignment on Image-Caption Data
In **Stage 1 (Feature Alignment)**, the vision encoder and LLM are kept completely **frozen** ($100\%$ parameter retention) [cite: 16]. Only the **projection MLP** is updated on image-caption pairs (such as MS-COCO) so the LLM learns to perceive visual tokens as natural semantic prompts [cite: 16].

In [ ]:
# Install required dependencies if in Colab/Jupyter
try:
    import transformers
    import datasets
except ImportError:
    !pip install -q transformers datasets torch torchvision accelerate pillow matplotlib ipywidgets

import os
import io
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, CLIPVisionModel, CLIPImageProcessor
from IPython.display import display, clear_output
import ipywidgets as widgets
from ipywidgets import interact, Dropdown, Text, IntSlider

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing on device: {device}")

torch.manual_seed(42)

## 📦 2. Loading Real Image-Caption Data (MS-COCO)
We load a compact slice of **real MS-COCO captions** using the Hugging Face `datasets` library. 
Each instance provides a natural photograph spanning everyday scenes (people, vehicles, animals, indoors) paired with descriptive human reference captions [cite: 16].

In [ ]:
NUM_SAMPLES = 250

print(f"Streaming {NUM_SAMPLES} real image-caption pairs from MS-COCO...")
try:
    coco_dataset = load_dataset("yokor/coco-captions", split=f"train[:{NUM_SAMPLES}]")
except Exception as e:
    print(f"Notice loading yokor/coco-captions: {e}. Trying alternative dataset slice...")
    coco_dataset = load_dataset("HuggingFaceM4/COCO", split=f"train[:{NUM_SAMPLES}]")

# Helper to normalize caption format across dataset variants
def extract_caption(example):
    raw = example.get("caption", example.get("sentences", ""))
    if isinstance(raw, list):
        # Pick the first caption string or raw dictionary entry
        first = raw[0]
        if isinstance(first, dict):
            return first.get("raw", str(first))
        return str(first)
    return str(raw)

print(f"Successfully loaded {len(coco_dataset)} real COCO examples!")

# Visualize a sample grid of 3 real COCO images and captions
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for idx in range(3):
    sample = coco_dataset[idx]
    img = sample["image"].convert("RGB")
    cap = extract_caption(sample)
    axes[idx].imshow(img)
    axes[idx].set_title(f"COCO #{idx}:\n{cap[:45]}...", fontsize=9)
    axes[idx].axis("off")
plt.tight_layout()
plt.show()

## 🏗️ 3. Initializing Models & Freezing Foundation Backbones
We build our Mini-LLaVA architecture using:
* **Vision Encoder:** `openai/clip-vit-base-patch32` ($d_v = 768$) [cite: 16]
* **Language Model:** `gpt2` ($d_{\text{LLM}} = 768$) [cite: 16]
* **Trainable Projector:** A 2-layer MLP with GELU activation ($768 \to 768 \to 768$) [cite: 16]

Both foundation models are frozen; only the projection bridge will receive gradient updates [cite: 16].

In [ ]:
VISION_MODEL_ID = "openai/clip-vit-base-patch32"
LLM_MODEL_ID = "gpt2"

print("Loading CLIP vision backbone & image processor...")
image_processor = CLIPImageProcessor.from_pretrained(VISION_MODEL_ID)
vision_encoder = CLIPVisionModel.from_pretrained(VISION_MODEL_ID).to(device)

print("Loading LLM tokenizer & causal model...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL_ID).to(device)

# Freeze both foundation backbones
for param in vision_encoder.parameters():
    param.requires_grad = False
for param in llm.parameters():
    param.requires_grad = False

# Define LLaVA-style 2-layer MLP projection bridge
class LLaVAMultimodalProjector(nn.Module):
    def __init__(self, vision_dim=768, llm_dim=768):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(vision_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim)
        )
        
    def forward(self, visual_features):
        return self.mlp(visual_features)

class MiniLLaVA(nn.Module):
    def __init__(self, vision_encoder, llm, projector):
        super().__init__()
        self.vision_encoder = vision_encoder
        self.llm = llm
        self.projector = projector
        
    def forward(self, pixel_values, text_input_ids, labels=None):
        # 1. Vision feature extraction: [B, P, D_v]
        vis_outputs = self.vision_encoder(pixel_values=pixel_values)
        vis_features = vis_outputs.last_hidden_state
        
        # 2. Project visual patch tokens: [B, P, D_llm]
        visual_tokens = self.projector(vis_features)
        
        # 3. Text word embeddings: [B, S, D_llm]
        text_embeds = self.llm.transformer.wte(text_input_ids)
        
        # 4. Interleave multimodal sequence: [Visual Tokens; Text Tokens]
        combined_embeds = torch.cat([visual_tokens, text_embeds], dim=1)
        
        # 5. Compute causal loss if labels are supplied
        if labels is not None:
            # Label masking: set -100 for all visual patch positions so
            # the model is only evaluated on generating text tokens
            num_vis_patches = visual_tokens.size(1)
            vis_ignore_labels = torch.full((labels.size(0), num_vis_patches), -100, dtype=torch.long, device=labels.device)
            full_labels = torch.cat([vis_ignore_labels, labels], dim=1)
            
            outputs = self.llm(inputs_embeds=combined_embeds, labels=full_labels)
            return outputs.loss, outputs.logits
        else:
            outputs = self.llm(inputs_embeds=combined_embeds)
            return outputs.logits

projector = LLaVAMultimodalProjector(vision_dim=768, llm_dim=768).to(device)
vlm_model = MiniLLaVA(vision_encoder, llm, projector)

trainable_params = sum(p.numel() for p in vlm_model.parameters() if p.requires_grad)
frozen_params = sum(p.numel() for p in vlm_model.parameters() if not p.requires_grad)
print(f"Trainable Parameters (MLP Projector ONLY): {trainable_params:,}")
print(f"Frozen Parameters (CLIP-ViT + GPT-2)     : {frozen_params:,}")

## 🏋️ 4. Training the Projector on Real COCO Captions
We now pre-process the COCO dataset and train the projector on predicting human descriptions from visual patch embeddings [cite: 16].

In [ ]:
class COCOMiniDataset(Dataset):
    def __init__(self, dataset, processor, tokenizer, max_samples=250):
        self.samples = []
        for i in range(min(max_samples, len(dataset))):
            item = dataset[i]
            try:
                img = item["image"].convert("RGB")
                cap = extract_caption(item)
                if len(cap.strip()) > 3:
                    self.samples.append((img, cap))
            except Exception:
                continue
                
        self.processor = processor
        self.tokenizer = tokenizer
        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        img, cap = self.samples[idx]
        # Preprocess image for CLIP (224x224)
        pixels = self.processor(images=img, return_tensors="pt").pixel_values.squeeze(0)
        # Format instruction prompt
        prompt = f"A photo of: {cap}"
        tokenized = self.tokenizer(prompt, return_tensors="pt", max_length=24, padding="max_length", truncation=True)
        
        return {
            "pixel_values": pixels,
            "input_ids": tokenized.input_ids.squeeze(0)
        }

train_coco = COCOMiniDataset(coco_dataset, image_processor, tokenizer, max_samples=200)
train_loader = DataLoader(train_coco, batch_size=16, shuffle=True)

# Training configuration
optimizer = torch.optim.AdamW(projector.parameters(), lr=2e-3, weight_decay=1e-2)
EPOCHS = 8

print(f"=== Training Multimodal Projector on {len(train_coco)} Real COCO Samples ===")
vlm_model.train()
loss_history = []

for epoch in range(1, EPOCHS + 1):
    running_loss = 0.0
    for batch in train_loader:
        b_pixels = batch["pixel_values"].to(device)
        b_ids = batch["input_ids"].to(device)
        
        optimizer.zero_grad()
        loss, _ = vlm_model(pixel_values=b_pixels, text_input_ids=b_ids, labels=b_ids)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * b_pixels.size(0)
        
    epoch_loss = running_loss / len(train_coco)
    loss_history.append(epoch_loss)
    print(f"Epoch {epoch:2d}/{EPOCHS} | Multimodal Next-Token Loss: {epoch_loss:.4f}")

# Plot convergence
plt.figure(figsize=(7, 3))
plt.plot(range(1, EPOCHS + 1), loss_history, marker='o', color='#1f77b4', linewidth=2)
plt.title("LLaVA Projector Alignment Loss on COCO", fontweight='bold')
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## 🎛️ 5. Interactive Multimodal Inference Lab
Select any real COCO test image below and enter a prompt prefix (e.g., `"A photo of:"`). 
The model will extract visual patch tokens, pass them through the trained projector bridge, and generate text autoregressively [cite: 16]!

In [ ]:
@torch.no_grad()
def run_interactive_vlm(sample_idx=5, prompt_prefix="A photo of:", max_new_tokens=15):
    vlm_model.eval()
    
    # Get test sample
    raw_item = coco_dataset[sample_idx]
    test_image = raw_item["image"].convert("RGB")
    gt_caption = extract_caption(raw_item)
    
    # Preprocess image
    pixels = image_processor(images=test_image, return_tensors="pt").pixel_values.to(device)
    
    # Project visual patches
    vis_features = vision_encoder(pixel_values=pixels).last_hidden_state
    visual_tokens = projector(vis_features)
    
    # Tokenize input text prefix
    input_ids = tokenizer(prompt_prefix, return_tensors="pt").input_ids.to(device)
    generated_ids = input_ids.clone()
    
    # Autoregressive generation loop
    for _ in range(max_new_tokens):
        text_embeds = llm.transformer.wte(generated_ids)
        combined = torch.cat([visual_tokens, text_embeds], dim=1)
        
        logits = llm(inputs_embeds=combined).logits[:, -1, :]
        next_token = torch.argmax(logits, dim=-1, keepdim=True)
        generated_ids = torch.cat([generated_ids, next_token], dim=1)
        
        if next_token.item() == tokenizer.eos_token_id:
            break
            
    model_output = tokenizer.decode(generated_ids[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
    
    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(test_image)
    ax.axis("off")
    plt.show()
    
    print(f"COCO Image Index   : #{sample_idx}")
    print(f"Ground-Truth Caption: {gt_caption}")
    print("-" * 60)
    print(f"Input Text Prompt  : '{prompt_prefix}'")
    print(f"VLM Continuation   : '{model_output}'")
    print("-" * 60)

interact(run_interactive_vlm,
         sample_idx=IntSlider(value=10, min=0, max=min(NUM_SAMPLES - 1, 150), step=1, description="COCO Index:"),
         prompt_prefix=Text(value="A photo of:", description="Prompt:"),
         max_new_tokens=IntSlider(value=12, min=5, max=30, step=1, description="Max Tokens:"));